In [5]:
import pandas as pd
import re, os
import subprocess
from tqdm.notebook import tqdm
from concurrent.futures import ProcessPoolExecutor

In [20]:
def generate_data(df_ag, df_ngs, train_list=None, train_size=100):
    if train_list:
        df_ag_use = df_ag.loc[train_list,:]
        df_ngs_use = df_ngs.loc[train_list,:]
        
        train_ag = df_ag_use.sample(train_size)
        train_index=train_ag.index
        train_ngs = df_ngs_use.loc[train_index,:]
    else:
        train_ag = df_ag.sample(train_size)
        train_index=train_ag.index
        train_ngs = df_ngs.loc[train_index,:]

    test_ag = df_ag.loc[~df_ag.index.isin(train_index),:]
    test_ngs = df_ngs.loc[~df_ngs.index.isin(train_index),:]
    return train_ag, test_ag, train_ngs, test_ngs

In [7]:
exp_id = 'BRCA_LUSC'
# Load processed datasets, Microarray and RNA-seq
ag_file = f'datasets/TCGA/{exp_id}/COMB_ag.tsv' # LUSC1
ngs_file = f'datasets/TCGA/{exp_id}/COMB_rs.tsv'
df_ag = pd.read_csv(ag_file,sep="\t",index_col=0).fillna('')
df_ngs = pd.read_csv(ngs_file,sep="\t",index_col=0).fillna('')

In [ ]:
exp_id = 'NCI60'
# Load processed datasets, Microarray and RNA-seq
ag_file = f'datasets/NCI60/NCI603_ag.tsv'
ngs_file = f'datasets/NCI60/NCI603_rs.tsv'
df_ag = pd.read_csv(ag_file,sep="\t",index_col=0).fillna('')
df_ngs = pd.read_csv(ngs_file,sep="\t",index_col=0).fillna('')

In [8]:
print(df_ag.shape, df_ngs.shape)
print(df_ag.max().max(), df_ngs.max().max())
print(df_ag.min().min(), df_ngs.min().min())

(739, 15623) (739, 15623)
11.055625 20.71859224019692
-10.2924 0.0


In [18]:
with open('datasets/TCGA/BRCA_LUSC/BRCA_samples.txt', "r") as f:
    train_list = f.readlines()
train_list = [x.strip('\n') for x in train_list]

In [19]:
exp_id = 'BRCA_LUSC'
ckp_dir = f'./run_logs/run_{exp_id}/'
if not os.path.exists(ckp_dir):
    os.system(f'mkdir {ckp_dir}')

In [27]:
def prepare_data_and_train(t, r, n, datamode, main_data_dir, ckp_dir, venv_python, train_list=None):
    # Generate the temporary data folder for the current analysis
    tmp_folder = f'tmp_{t}_{r}/'
    tmp_path = os.path.join(main_data_dir, tmp_folder)
    if os.path.exists(tmp_path):
        os.system(f"rm -rf {tmp_path}")
    os.makedirs(tmp_path, exist_ok=True)

    for sub_folder in ['trainAG/', 'trainNGS/', 'testAG/', 'testNGS/']:
        curr_folder = os.path.join(tmp_path, sub_folder)
        os.makedirs(curr_folder, exist_ok=True)

    train_ag, test_ag, train_ngs, test_ngs = generate_data(df_ag, df_ngs, train_list, train_size=t)

    # Save training and testing data
    for dataset, subdir in [(train_ag, 'trainAG/'), (train_ngs, 'trainNGS/'),
                            (test_ag, 'testAG/'), (test_ngs, 'testNGS/')]:
        for ind, d in dataset.iterrows():
            pure_list = d.tolist()
            with open(os.path.join(tmp_path, subdir, f'{ind}.txt'), 'w') as f:
                f.write("\t".join(map(str, pure_list)))

    # Construct the command for training
    
    command = [
        venv_python, 'train_new.py',
        '-t', str(t), '-r', str(r),
        '--dataroot', str(tmp_path),
        '-n', str(n), '--dataset_mode', str(datamode),
        '--ckp_dir', str(ckp_dir),
        '--n_epochs', '5000', '--n_epochs_decay', '500',
        '--gpu_ids', '7',
        '--input_nc', str(train_ag.shape[1]),
        '--output_nc', str(train_ag.shape[1]),
        '--save_epoch_freq', '500'
    
    ]
    '''
    command = [venv_python, '--version']
    '''
    
    # Execute the training command
    result = subprocess.run(command, capture_output=True, text=True)
    return result.stdout, result.stderr

In [ ]:
# Parameters
max_workers = 1
venv_python = '/main/projects/GANomics/venv/bin/python'
main_data_dir = f'run_datasets/datasets_{exp_id}/'
num_runs = 1
train_samples_size = [50]
n = 'FEEDBACKv2'
d = 'unalignedpaired'

# Create a list of tasks
tasks = [
    (t, r, n, d, main_data_dir, ckp_dir, venv_python, train_list)
    for t in train_samples_size
    for r in range(1, num_runs+1)
]

# Execute tasks in parallel
def task_wrapper(params):
    return prepare_data_and_train(*params)
    
for t in tasks:
    result = task_wrapper(t)

In [28]:
print(result[0])

----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
                  ckp_dir: ./run_logs/run_BRCA_LUSC/     	[default: run/]
           continue_train: False                         
                crop_size: 256                           
                 dataroot: run_datasets/datasets_BRCA_LUSC/tmp_50_0/	[default: datasets/]
             dataset_mode: unalignedpaired               	[default: unaligned]
                 datatype: text                          
                direction: AtoB                          
              display_env: main                          
             display_freq: 400                           
               display_id: 1                             
            display_ncols: 4                             
             display_port: 8090                          
           display_

In [4]:
## Batched analyses
# Parameters

experiments = ['BRCA2','LAML1','LUSC1']
for exp_id in experiments:
    print(f"Working on {exp_id}.")

    max_workers = 1
    venv_python = '/main/projects/GANomics/venv/bin/python'
    main_data_dir = f'run_datasets/datasets_{exp_id}/'
    num_runs = 1
    train_samples_size = [10,]
    n = 'FEEDBACKv2'
    d = 'unalignedpaired'
    
    exp = re.sub(r'\d+','',exp_id)
    # Load processed datasets, Microarray and RNA-seq
    ag_file = f'datasets/TCGA/{exp}/{exp_id}_ag.tsv'
    ngs_file = f'datasets/TCGA/{exp}/{exp_id}_rs.tsv'
    df_ag = pd.read_csv(ag_file,sep="\t",index_col=0).fillna('')
    df_ngs = pd.read_csv(ngs_file,sep="\t",index_col=0).fillna('')

    ckp_dir = f'./run_logs/run_{exp_id}/'
    if not os.path.exists(ckp_dir):
        os.system(f'mkdir {ckp_dir}')
    
    # Create a list of tasks
    tasks = [
        (t, r, n, d, main_data_dir, ckp_dir, venv_python)
        for t in train_samples_size
        for r in range(0, num_runs)
    ]
    
    # Execute tasks in parallel
    def task_wrapper(params):
        return prepare_data_and_train(*params)
        
    for t in tasks:
        result = task_wrapper(t)

Working on BRCA2.
Working on LAML1.
Working on LUSC1.


In [ ]:
print(result[1])